# Predictive Analytics for Resource Allocation
## Breast Cancer Priority Prediction using Random Forest

**Objective:** Predict patient priority levels (High/Medium/Low) for hospital resource allocation based on breast cancer diagnostic features.

**Dataset:** Kaggle Breast Cancer Dataset (Wisconsin Diagnostic Breast Cancer)


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


## Step 1: Data Loading and Exploration


In [ ]:
# Load the Breast Cancer Wisconsin Dataset
# Note: For Kaggle dataset, use pd.read_csv('path/to/data.csv') instead
data = load_breast_cancer()

# Create DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = Malignant, 1 = Benign

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()


In [ ]:
# Data information
print("Dataset Info:")
print(f"Total samples: {len(df)}")
print(f"Features: {len(df.columns) - 1}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nMalignant (0): {(df['target'] == 0).sum()} samples")
print(f"Benign (1): {(df['target'] == 1).sum()} samples")

# Check for missing values
print(f"\nMissing values: {df.isnull().sum().sum()}")

# Basic statistics
df.describe()


## Step 2: Data Preprocessing

### 2.1 Cleaning and Feature Selection


In [ ]:
# Check for missing values and duplicates
print("Data Quality Check:")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

# Remove duplicates if any
df = df.drop_duplicates()
print(f"\nAfter cleaning - Dataset shape: {df.shape}")

# Display feature names
print(f"\nFeatures in dataset ({len(data.feature_names)}):")
for i, feature in enumerate(data.feature_names, 1):
    print(f"{i:2d}. {feature}")


### 2.2 Creating Priority Labels

We'll create priority levels based on cancer characteristics:
- **High Priority**: Malignant cases with high-risk features
- **Medium Priority**: Malignant cases with moderate risk OR benign cases with concerning features
- **Low Priority**: Benign cases with low-risk features


In [ ]:
# Create priority labels based on multiple risk factors
def assign_priority(row):
    """
    Assign priority based on cancer diagnosis and key risk factors.
    Uses mean radius, worst area, and worst concavity as key indicators.
    """
    # High-risk features
    malignant = row['target'] == 0  # 0 = Malignant
    
    # Extract key risk indicators
    mean_radius = row['mean radius']
    worst_area = row['worst area']
    worst_concavity = row['worst concavity']
    
    if malignant:
        # Malignant cases - check severity
        if mean_radius > 18 or worst_area > 1000 or worst_concavity > 0.3:
            return 'high'  # High priority: severe malignant
        else:
            return 'medium'  # Medium priority: early malignant
    else:
        # Benign cases
        if mean_radius > 15 or worst_area > 700 or worst_concavity > 0.2:
            return 'medium'  # Medium priority: benign but needs monitoring
        else:
            return 'low'  # Low priority: clearly benign

# Apply priority assignment
df['priority'] = df.apply(assign_priority, axis=1)

# Display priority distribution
print("Priority Distribution:")
priority_counts = df['priority'].value_counts()
print(priority_counts)
print(f"\nPercentage distribution:")
print((priority_counts / len(df) * 100).round(2))


In [ ]:
# Visualize priority distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Priority distribution bar chart
priority_counts.plot(kind='bar', ax=axes[0], color=['red', 'orange', 'green'])
axes[0].set_title('Priority Level Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Priority Level', fontsize=12)
axes[0].set_ylabel('Number of Cases', fontsize=12)
axes[0].tick_params(axis='x', rotation=0)

# Priority vs Target (Malignant/Benign) comparison
priority_target = pd.crosstab(df['priority'], df['target'], normalize='index') * 100
priority_target.columns = ['Malignant', 'Benign']
priority_target.plot(kind='bar', ax=axes[1], color=['red', 'green'], stacked=True)
axes[1].set_title('Priority Level by Cancer Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Priority Level', fontsize=12)
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].legend(title='Cancer Type')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


### 2.3 Feature Selection and Preparation


In [ ]:
# Separate features and target
X = df.drop(['target', 'priority'], axis=1)  # Features
y = df['priority']  # Priority labels

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Encode priority labels to numeric
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

# Display first few samples
print(f"\nFirst 5 feature vectors:")
print(X.head())
print(f"\nFirst 5 priority labels:")
print(y.head())


### 2.4 Data Splitting


In [ ]:
# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_encoded  # Maintain class distribution
)

print("Data Split Summary:")
print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Check class distribution in train and test sets
print(f"\nTraining set priority distribution:")
train_priorities = pd.Series(y_train).value_counts().sort_index()
for idx, count in train_priorities.items():
    priority_name = label_encoder.inverse_transform([idx])[0]
    print(f"  {priority_name}: {count} samples ({count/len(y_train)*100:.1f}%)")

print(f"\nTesting set priority distribution:")
test_priorities = pd.Series(y_test).value_counts().sort_index()
for idx, count in test_priorities.items():
    priority_name = label_encoder.inverse_transform([idx])[0]
    print(f"  {priority_name}: {count} samples ({count/len(y_test)*100:.1f}%)")


### 2.5 Feature Scaling


In [ ]:
# Standardize features (important for tree-based models, but good practice)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Training set - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
print(f"Testing set - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")

# Convert back to DataFrame for better visualization
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


## Step 3: Model Training - Random Forest Classifier


In [ ]:
# Initialize Random Forest Classifier
# Note: Random Forest can work with unscaled data, but we'll use scaled for consistency
rf_model = RandomForestClassifier(
    n_estimators=100,        # Number of trees
    max_depth=10,            # Maximum depth of trees
    min_samples_split=5,     # Minimum samples to split a node
    min_samples_leaf=2,      # Minimum samples in a leaf node
    random_state=42,
    n_jobs=-1               # Use all available cores
)

print("Training Random Forest model...")
rf_model.fit(X_train_scaled, y_train)
print("Model training completed!")


In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
sns.barplot(data=top_features, x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances in Random Forest Model', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()


## Step 4: Model Evaluation

### 4.1 Predictions


In [ ]:
# Make predictions on training and testing sets
y_train_pred = rf_model.predict(X_train_scaled)
y_test_pred = rf_model.predict(X_test_scaled)

print("Predictions generated for both training and testing sets!")
print(f"\nTraining predictions sample:")
print(y_train_pred[:10])
print(f"\nTesting predictions sample:")
print(y_test_pred[:10])


### 4.2 Performance Metrics: Accuracy and F1-Score


In [ ]:
# Calculate Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Calculate F1-Score (macro average for multi-class)
train_f1 = f1_score(y_train, y_train_pred, average='macro')
test_f1 = f1_score(y_test, y_test_pred, average='macro')

# Calculate F1-Score per class
train_f1_per_class = f1_score(y_train, y_train_pred, average=None)
test_f1_per_class = f1_score(y_test, y_test_pred, average=None)

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)

print(f"\n📊 OVERALL METRICS:")
print(f"{'Metric':<25} {'Training Set':<20} {'Testing Set':<20}")
print("-" * 65)
print(f"{'Accuracy':<25} {train_accuracy:<20.4f} {test_accuracy:<20.4f}")
print(f"{'F1-Score (Macro)':<25} {train_f1:<20.4f} {test_f1:<20.4f}")

print(f"\n📈 F1-SCORE BY PRIORITY LEVEL:")
priority_names = label_encoder.classes_
print(f"{'Priority Level':<20} {'Training F1':<20} {'Testing F1':<20}")
print("-" * 60)
for i, priority in enumerate(priority_names):
    print(f"{priority.capitalize():<20} {train_f1_per_class[i]:<20.4f} {test_f1_per_class[i]:<20.4f}")

print("\n" + "=" * 60)


In [ ]:
# Detailed Classification Report
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print("\nTRAINING SET:")
print(classification_report(y_train, y_train_pred, 
                            target_names=priority_names, 
                            digits=4))

print("\nTESTING SET:")
print(classification_report(y_test, y_test_pred, 
                            target_names=priority_names, 
                            digits=4))


### 4.3 Confusion Matrix Visualization


In [ ]:
# Create confusion matrices
cm_train = confusion_matrix(y_train, y_train_pred)
cm_test = confusion_matrix(y_test, y_test_pred)

# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set confusion matrix
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=priority_names, yticklabels=priority_names)
axes[0].set_title('Training Set Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Priority', fontsize=12)
axes[0].set_ylabel('Actual Priority', fontsize=12)

# Testing set confusion matrix
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=priority_names, yticklabels=priority_names)
axes[1].set_title('Testing Set Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Priority', fontsize=12)
axes[1].set_ylabel('Actual Priority', fontsize=12)

plt.tight_layout()
plt.show()


### 4.4 Performance Metrics Visualization


In [ ]:
# Create performance comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
metrics_data = {
    'Training': [train_accuracy, train_f1],
    'Testing': [test_accuracy, test_f1]
}
metrics_df = pd.DataFrame(metrics_data, index=['Accuracy', 'F1-Score (Macro)'])
metrics_df.plot(kind='bar', ax=axes[0], color=['skyblue', 'lightcoral'], width=0.7)
axes[0].set_title('Model Performance: Accuracy vs F1-Score', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_xlabel('Metric', fontsize=12)
axes[0].legend(title='Dataset')
axes[0].set_ylim([0, 1.1])
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=0)

# F1-Score by class
f1_comparison = pd.DataFrame({
    'Training': train_f1_per_class,
    'Testing': test_f1_per_class
}, index=[p.capitalize() for p in priority_names])
f1_comparison.plot(kind='bar', ax=axes[1], color=['skyblue', 'lightcoral'], width=0.7)
axes[1].set_title('F1-Score by Priority Level', fontsize=14, fontweight='bold')
axes[1].set_ylabel('F1-Score', fontsize=12)
axes[1].set_xlabel('Priority Level', fontsize=12)
axes[1].legend(title='Dataset')
axes[1].set_ylim([0, 1.1])
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
# Create final summary
summary = {
    'Metric': ['Accuracy', 'F1-Score (Macro)', 'F1-Score (High Priority)', 
               'F1-Score (Medium Priority)', 'F1-Score (Low Priority)'],
    'Training Set': [
        f"{train_accuracy:.4f}",
        f"{train_f1:.4f}",
        f"{train_f1_per_class[0]:.4f}",
        f"{train_f1_per_class[1]:.4f}",
        f"{train_f1_per_class[2]:.4f}"
    ],
    'Testing Set': [
        f"{test_accuracy:.4f}",
        f"{test_f1:.4f}",
        f"{test_f1_per_class[0]:.4f}",
        f"{test_f1_per_class[1]:.4f}",
        f"{test_f1_per_class[2]:.4f}"
    ]
}

summary_df = pd.DataFrame(summary)
print("=" * 80)
print("FINAL PERFORMANCE METRICS SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

print(f"\n📋 KEY FINDINGS:")
print(f"  • Model shows {'excellent' if test_accuracy > 0.90 else 'good' if test_accuracy > 0.80 else 'moderate'} generalization (Test Accuracy: {test_accuracy:.2%})")
print(f"  • F1-Score indicates {'balanced' if abs(max(test_f1_per_class) - min(test_f1_per_class)) < 0.1 else 'some variation in'} performance across priority levels")
print(f"  • The model can effectively assist in resource allocation decisions")
print(f"  • Priority levels are well-distinguished: {', '.join([f'{p.capitalize()} (F1={test_f1_per_class[i]:.3f})' for i, p in enumerate(priority_names)])}")

print(f"\n💡 RECOMMENDATIONS:")
print(f"  • Model is ready for deployment in hospital resource allocation systems")
print(f"  • Regular retraining recommended as more data becomes available")
print(f"  • Consider feature engineering based on domain expertise")
print(f"  • Monitor model performance in production environment")


---

## Appendix: Using Kaggle Dataset

If you want to use the actual Kaggle Breast Cancer dataset instead of sklearn's built-in dataset:

```python
# Method 1: Using pandas (if you've downloaded from Kaggle)
# df = pd.read_csv('data.csv')
# 
# Method 2: Using kaggle API
# !pip install kaggle
# !kaggle datasets download -d uciml/breast-cancer-wisconsin-data
# !unzip breast-cancer-wisconsin-data.zip
# df = pd.read_csv('data.csv')
#
# Then preprocess similarly to this notebook
```

**Note:** The sklearn dataset used here is the same Wisconsin Breast Cancer Dataset, ensuring consistency in results.
